# DRBO — Formeln und Implementierung

Referenznotebook: jede Formel aus Kirschner et al. (2020), *"Distributionally Robust Bayesian Optimization"*, direkt neben dem Code aus `drbo.py`.

Muster: **Markdown-Zelle mit LaTeX** (`$$...$$`) &rarr; **Code-Zelle direkt darunter**. Beim Ausfuehren rendert Jupyter die Formeln automatisch ueber MathJax.

In [ ]:
import numpy as np
import scipy.optimize

## 1. MMD-Kernel und Gram-Matrix (Paper Sec. 2)

RBF-Kernel ueber Kontextpunkte:

$$k_M(c_i, c_j) = \exp\left(-\dfrac{\|c_i - c_j\|^2}{2\ell^2}\right)$$

Die quadrierte MMD-Distanz zwischen zwei diskreten Verteilungen $w, w'$ ueber denselben Kandidatenpunkten:

$$d(P,Q)^2 = (w-w')^\top M (w-w'), \qquad M_{ij} = k_M(c_i,c_j)$$

In [ ]:
def _mmd_gram_matrix(context_candidates, mmd_lengthscale):
    """RBF-Kernel-Gram-Matrix k_M(c_i, c_j) ueber diskrete Kontext-Kandidaten."""
    diffs = context_candidates[:, None, :] - context_candidates[None, :, :]
    sqdist = np.sum(diffs ** 2, axis=-1)
    return np.exp(-0.5 * sqdist / mmd_lengthscale ** 2)

# Kleines Beispiel: 5 skalare Kontextkandidaten
demo_contexts = np.array([[0.1], [0.3], [0.5], [0.7], [0.9]])
M = _mmd_gram_matrix(demo_contexts, mmd_lengthscale=0.3)
M

## 2. Inneres worst-case-Problem (Algorithm 1, Zeile 3)

$$w^{ucb}_x = \underset{w'}{\operatorname{argmin}} \ \langle \mathrm{ucb}_x, w' \rangle \quad
\text{s.t.} \quad \sum_j w'_j = 1,\ \ 0 \le w'_j \le 1,\ \ \|w' - w_{ref}\|_M \le \varepsilon$$

In [ ]:
def _solve_worst_case_weights(ucb_values, w_ref, M, epsilon):
    """Loest das konvexe Programm aus Algorithm 1, Zeile 3."""
    n = len(ucb_values)
    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: epsilon ** 2 - (w - w_ref) @ M @ (w - w_ref)},
    ]
    bounds = [(0.0, 1.0)] * n
    res = scipy.optimize.minimize(
        lambda w: w @ ucb_values, w_ref.copy(),
        method="SLSQP", bounds=bounds, constraints=constraints,
    )
    return res.x if res.success else w_ref

# Beispiel: gleichverteilte Referenz, willkuerliche UCB-Werte
w_ref_demo = np.ones(5) / 5
ucb_demo = np.array([1.0, 2.5, 0.5, 3.0, 1.8])
w_worst = _solve_worst_case_weights(ucb_demo, w_ref_demo, M, epsilon=0.3)
w_worst

## 3. Empirische Referenzverteilung und Radius (Corollary 4 / Lemma 3)

$$\hat P_t = \frac{1}{t-1} \sum_{s<t} \delta_{c_s}
\qquad\qquad
\varepsilon_t = \frac{2 + \sqrt{2\log(1/\delta)}}{\sqrt{t-1}}$$

In [ ]:
def epsilon_t(n_obs, delta=0.1):
    return (2.0 + np.sqrt(2.0 * np.log(1.0 / delta))) / np.sqrt(max(n_obs, 1))

for t in [2, 5, 10, 50, 200]:
    print(f"t={t:>4}  eps_t={epsilon_t(t):.4f}")

## 4. UCB und aeusseres Maximum (Algorithm 1, Zeile 4)

$$\mathrm{ucb}_t(x,c) = \hat f_t(x,c) + \sqrt{\beta_t}\,\sigma_t(x,c)
\qquad\qquad
x_t = \underset{x}{\operatorname{argmax}} \ \langle w^{ucb}_x, \mathrm{ucb}_x \rangle$$

## 5. Kontextwahl im Simulator-Setting (Theorem 5)

$$c_t = \underset{c}{\operatorname{argmax}} \ \sigma_t(x_t, c)$$

*(zum Vergleich: `drbo.py` zieht `next_context` stattdessen zufaellig aus der worst-case-Verteilung `w_worst` — siehe Diskussion im Chat / Artifact.)*